In [1]:
!pip install -q langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/ufc-rag-proj')

Mounted at /content/drive


In [3]:
import pandas as pd
from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/ufc-rag-proj")
RAW_CSV = PROJECT_ROOT / "data/ufc_fights_clean.csv"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
DB_PATH = PROJECT_ROOT / "data/ufc.duckdb"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


df = pd.read_csv(RAW_CSV)
df.shape


/tmp/ipykernel_803/3168286596.py:11: DtypeWarning: Columns (67,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_CSV)


(8584, 95)

In [4]:
df.columns

Index(['fight_outcome', 'outcome', 'event_date', 'event_name',
       'event_location', 'fighter_a_name', 'fighter_b_name', 'weight_class',
       'round', 'time', 'time_format', 'bout_type', 'referee', 'bonus',
       'details', 'fighter_a_age', 'fighter_b_age', 'age_diff',
       'fighter_a_reach', 'fighter_b_reach', 'reach_diff', 'fighter_a_height',
       'fighter_b_height', 'fighter_a_weight', 'fighter_b_weight',
       'fighter_a_sig_strikes_landed', 'fighter_b_sig_strikes_landed',
       'fighter_a_sig_strikes_attempted', 'fighter_b_sig_strikes_attempted',
       'fighter_a_total_strikes_landed', 'fighter_b_total_strikes_landed',
       'fighter_a_total_strikes_attempted',
       'fighter_b_total_strikes_attempted', 'fighter_a_body_strikes_landed',
       'fighter_b_body_strikes_landed', 'fighter_a_body_strikes_attempted',
       'fighter_b_body_strikes_attempted', 'fighter_a_clinch_strikes_landed',
       'fighter_b_clinch_strikes_landed', 'fighter_a_clinch_strikes_attempted',


In [5]:
##CREATING THE EVENTS TABLe


events = (
    df[['event_id','event_name', 'event_date','event_location']]
    .drop_duplicates(subset=['event_id'])
    .copy()
)

events = events.rename(columns={
    'event_name':"name",
    "event_date": "date",
    "event_location":"location"
})

events['date'] = pd.to_datetime(events['date'], errors="coerce")
print(events.shape)
print(f"from {events['date'].min()} to {events['date'].max()}")
events.head()


(768, 4)
from 1994-03-11 00:00:00 to 2026-04-04 00:00:00


,event_id,name,date,location
0,ufc-2-no-way-out,UFC 2: No Way Out,1994-03-11,"Denver, Colorado, USA"
15,ufc-3-the-american-dream,UFC 3: The American Dream,1994-09-09,"Charlotte, North Carolina, USA"
21,ufc-4-revenge-of-the-warriors,UFC 4: Revenge of the Warriors,1994-12-16,"Tulsa, Oklahoma, USA"
29,ufc-5-the-return-of-the-beast,UFC 5: The Return of the Beast,1995-04-07,"Charlotte, North Carolina, USA"
38,ufc-6-clash-of-the-titans,UFC 6: Clash of the Titans,1995-07-14,"Casper, Wyoming, USA"


In [6]:
df.columns

Index(['fight_outcome', 'outcome', 'event_date', 'event_name',
       'event_location', 'fighter_a_name', 'fighter_b_name', 'weight_class',
       'round', 'time', 'time_format', 'bout_type', 'referee', 'bonus',
       'details', 'fighter_a_age', 'fighter_b_age', 'age_diff',
       'fighter_a_reach', 'fighter_b_reach', 'reach_diff', 'fighter_a_height',
       'fighter_b_height', 'fighter_a_weight', 'fighter_b_weight',
       'fighter_a_sig_strikes_landed', 'fighter_b_sig_strikes_landed',
       'fighter_a_sig_strikes_attempted', 'fighter_b_sig_strikes_attempted',
       'fighter_a_total_strikes_landed', 'fighter_b_total_strikes_landed',
       'fighter_a_total_strikes_attempted',
       'fighter_b_total_strikes_attempted', 'fighter_a_body_strikes_landed',
       'fighter_b_body_strikes_landed', 'fighter_a_body_strikes_attempted',
       'fighter_b_body_strikes_attempted', 'fighter_a_clinch_strikes_landed',
       'fighter_b_clinch_strikes_landed', 'fighter_a_clinch_strikes_attempted',


In [7]:
#CREATING THE FIGHTERS TABLE


fighters_a  = df[['fighter_a_id', 'fighter_a_name', 'fighter_a_dob', 'fighter_a_height', 'fighter_a_reach']].copy()
fighters_a.columns = ['fighter_id', 'name', 'dob', 'height','reach']



fighters_b  = df[['fighter_b_id', 'fighter_b_name', 'fighter_b_dob', 'fighter_b_height', 'fighter_b_reach']].copy()
fighters_b.columns = ['fighter_id', 'name', 'dob', 'height','reach']

fighters = (
    pd.concat([fighters_a, fighters_b], ignore_index=True)
    .drop_duplicates(subset=['fighter_id'])
    .dropna(subset=['fighter_id'])
    .copy()
)

fighters['dob'] = pd.to_datetime(fighters['dob'], errors="coerce")
print(fighters.shape)
print(fighters['fighter_id'].nunique())
fighters = fighters.reset_index(drop=True)




(2685, 5)
2685


In [8]:
#CREATING THE FIGHTS TABLE

fights = df[[
    'fight_id',
    "event_id",
    "weight_class",
    "bout_type",
    "method",
    "round",
    "time",
    "time_format",
    "referee",
    "bonus",
    "details"
]].drop_duplicates(subset=['fight_id']).copy()

print(fights.shape)
print(fights['fight_id'].nunique())
fights.head()

(8584, 11)
8584


,fight_id,event_id,weight_class,bout_type,method,round,time,time_format,referee,bonus,details
0,00835554f95fa911,ufc-2-no-way-out,Open Weight,UFC 2 Tournament Title Bout,KO/TKO,1.0,1:17,No Time Limit,John McCarthy,belt,Punches to Head From Mount Submission to Strikes
1,aa161c7159741766,ufc-2-no-way-out,Open Weight,Open Weight Bout,Submission,1.0,1:31,No Time Limit,John McCarthy,-,Gi Choke From Back Control
2,5655639baecd8f6a,ufc-2-no-way-out,Open Weight,Open Weight Bout,Submission,1.0,1:07,No Time Limit,John McCarthy,-,Armbar From Mount
3,8ea72cf82fac07f6,ufc-2-no-way-out,Open Weight,Open Weight Bout,KO/TKO,1.0,0:30,No Time Limit,John McCarthy,-,Elbows to Head From Mount
4,b72acedda7ddc006,ufc-2-no-way-out,Open Weight,Open Weight Bout,Submission,1.0,3:07,No Time Limit,John McCarthy,-,Bulldog Choke On Ground


In [9]:
##CREATING THE WIDE FORMAT TABLE WHERE EACH ROW IS A FIGHT FOR EACH FIGHT OF A FIGHTER

def make_side(df, side: str):
  prefix = f"fighter_{side}_"
  opp = 'b' if side == 'a' else 'a'

  rename_map = {
        "fight_id": "fight_id",
        f"{prefix}id": "fighter_id",
        f"fighter_{opp}_id": "opponent_id",
        f"{prefix}sig_strikes_landed": "sig_strikes_landed",
        f"{prefix}sig_strikes_attempted": "sig_strikes_attempted",
        f"{prefix}total_strikes_landed": "total_strikes_landed",
        f"{prefix}total_strikes_attempted": "total_strikes_attempted",
        f"{prefix}takedowns_landed": "takedowns_landed",
        f"{prefix}takedowns_attempted": "takedowns_attempted",
        f"{prefix}submission_attempts": "submission_attempts",
        f"{prefix}knockdowns": "knockdowns",
        f"{prefix}reversals": "reversals",
        f"{prefix}control_time_per_fight": "control_time_seconds",
        f"{prefix}fight_minutes": "fight_minutes",
        f"{prefix}head_strikes_landed": "head_strikes_landed",
        f"{prefix}body_strikes_landed": "body_strikes_landed",
        f"{prefix}leg_strikes_landed": "leg_strikes_landed",
        f"{prefix}distance_strikes_landed": "distance_strikes_landed",
        f"{prefix}clinch_strikes_landed": "clinch_strikes_landed",
        f"{prefix}ground_strikes_landed": "ground_strikes_landed",
    }

  existing = {k:v for k,v in rename_map.items() if k in df.columns}
  side_df = df[list(existing.keys())].rename(columns=existing).copy()

  if side == 'a':
        side_df["is_winner"] = df["fight_outcome"] == 'red_win'
  else:
        side_df["is_winner"] = df['fight_outcome'] == 'blue_win'


  return side_df

perf_a = make_side(df, 'a')
perf_b = make_side(df, 'b')

fight_performances = pd.concat([perf_a, perf_b], ignore_index=True)




print(fight_performances.shape)
fight_performances['is_winner'].value_counts()

(17168, 21)


,count
is_winner,
False,8730
True,8438


In [10]:
events.to_parquet(PROCESSED_DIR / "events.parquet", index=False)
fighters.to_parquet(PROCESSED_DIR / "fighters.parquet", index=False)
fights.to_parquet(PROCESSED_DIR / "fights.parquet", index=False)
fight_performances.to_parquet(PROCESSED_DIR / 'fight_performances.parquet', index=False)


for f in PROCESSED_DIR.glob("*.parquet"):
  print(f)

/content/drive/MyDrive/ufc-rag-proj/data/processed/fight_performances.parquet
/content/drive/MyDrive/ufc-rag-proj/data/processed/events.parquet
/content/drive/MyDrive/ufc-rag-proj/data/processed/fighters.parquet
/content/drive/MyDrive/ufc-rag-proj/data/processed/fights.parquet


In [11]:
import duckdb


conn = duckdb.connect(str(DB_PATH))
conn.execute("CREATE OR REPLACE TABLE events AS SELECT * FROM read_parquet(?)", [str(PROCESSED_DIR / "events.parquet")])
conn.execute("CREATE OR REPLACE TABLE fighters AS SELECT * FROM read_parquet(?)", [str(PROCESSED_DIR / "fighters.parquet")])
conn.execute("CREATE OR REPLACE TABLE fights AS SELECT * FROM read_parquet(?)", [str(PROCESSED_DIR / "fights.parquet")])
conn.execute("CREATE OR REPLACE TABLE fight_performances AS SELECT * FROM read_parquet(?)", [str(PROCESSED_DIR / "fight_performances.parquet")])

conn.execute("CREATE INDEX IF NOT EXISTS idx_fp_fighter ON fight_performances(fighter_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_fp_fight ON fight_performances(fight_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_fp_events ON fight_performances(fighter_id)")

print("DuckDB tables created succesfully")
print(conn.execute("SHOW TABLES").fetchdf())

conn.close()

DuckDB tables created succesfully
                 name
0              events
1  fight_performances
2            fighters
3              fights


In [54]:
# check

conn = duckdb.connect(str(DB_PATH))

# print("most significant strikes (career) top 5 ")
# print(conn.execute("""
# SELECT f.name, SUM(fp.sig_strikes_landed) as total_sig_strikes
# FROM fight_performances fp
# JOIN fighters f ON fp.fighter_id = f.fighter_id
# GROUP BY 1
# ORDER BY 2 DESC
# LIMIT 5
# """).fetchdf())


print('fights at ufc 300')
print(conn.execute("""
SELECT COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'fight_performances';
""").fetchdf())

print(conn.execute("""
SELECT e.name as event_name, fa.name as fighter_a, fb.name as fighter_b, method, weight_class, fpa.is_winner
FROM fights fi
JOIN events e ON fi.event_id = e.event_id
JOIN fight_performances fpa on fi.fight_id = fpa.fight_id
JOIN fighters fa on fpa.fighter_id = fa.fighter_id
JOIN fight_performances fpb on fi.fight_id = fpb.fight_id
JOIN fighters fb on fpb.fighter_id = fb.fighter_id AND fb.fighter_id != fa.fighter_id
WHERE e.name ILIKE '%UFC 300%'




""").fetchdf())




fights at ufc 300
                column_name
0                  fight_id
1                fighter_id
2               opponent_id
3        sig_strikes_landed
4     sig_strikes_attempted
5      total_strikes_landed
6   total_strikes_attempted
7          takedowns_landed
8       takedowns_attempted
9       submission_attempts
10               knockdowns
11                reversals
12     control_time_seconds
13            fight_minutes
14      head_strikes_landed
15      body_strikes_landed
16       leg_strikes_landed
17  distance_strikes_landed
18    clinch_strikes_landed
19    ground_strikes_landed
20                is_winner
                   event_name            fighter_a            fighter_b  \
0   UFC 300: Pereira vs. Hill       RENATO MOICANO         JALIN TURNER   
1   UFC 300: Pereira vs. Hill         JAMAHAL HILL         ALEX PEREIRA   
2   UFC 300: Pereira vs. Hill          YAN XIAONAN          ZHANG WEILI   
3   UFC 300: Pereira vs. Hill        CODY BRUNDAGE            BO N